# Home Credit PD Scorecard Build

This notebook turns the earlier enhanced PD modelling work into a **bank-style scorecard workflow** for a GitHub portfolio.

## What this notebook does
- loads the Home Credit application data and external engineered features
- rebuilds a compact **scorecard candidate set**
- performs **WOE / IV transformation**
- fits a **logistic regression scorecard**
- converts model outputs into **points**
- saves scorecard artefacts for validation and business use

## Portfolio positioning
This notebook is designed to answer the interview question:

> "You ran logistic regression. Can you also turn that into a scorecard?"

The answer here is **yes**: the notebook converts a predictive model into a more governed and explainable scorecard structure.

## 1. Setup

This project assumes your working folders follow the same structure used in your earlier Home Credit project:

- `../data/application_train.csv`
- `../processed/home_credit_external_features.csv`

If you already saved a merged modelling base, the notebook will also use it.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data")
PROCESSED_DIR = Path("../processed")
OUTPUT_DIR = Path("../output/scorecard_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

APPLICATION_FILE = DATA_DIR / "application_train.csv"
EXTERNAL_FILE = PROCESSED_DIR / "home_credit_external_features.csv"
MERGED_FILE = PROCESSED_DIR / "home_credit_model_base_after_merge.csv"

RANDOM_STATE = 42
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

## 2. Load and prepare the modelling base

The earlier enhanced notebook merged application data with engineered bureau, instalment, POS and previous-loan features.

This notebook recreates that modelling base in a reusable way.

In [ ]:
def load_model_base(application_file=APPLICATION_FILE, external_file=EXTERNAL_FILE, merged_file=MERGED_FILE):
    if merged_file.exists():
        df = pd.read_csv(merged_file)
        print(f"Loaded merged modelling base: {merged_file}")
        return df

    application = pd.read_csv(application_file)
    print(f"Loaded application data: {application.shape}")

    if external_file.exists():
        external = pd.read_csv(external_file)
        print(f"Loaded external feature table: {external.shape}")
        df = application.merge(external, on=ID_COL, how="left")
        print(f"Merged model base: {df.shape}")
    else:
        df = application.copy()
        print("External feature file not found. Continuing with application-level features only.")

    return df

df = load_model_base()
df.head()

## 3. Recreate key application features

These are the core application-level variables used in the earlier PD work.  
They are simple, explainable, and useful for a scorecard build.

In [ ]:
def engineer_application_features(df):
    df = df.copy()

    if "DAYS_EMPLOYED" in df.columns:
        df["DAYS_EMPLOYED"] = pd.to_numeric(df["DAYS_EMPLOYED"], errors="coerce").replace(365243, np.nan)

    if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["DTI"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)

    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["INST_TO_INCOME"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)

    if {"AMT_CREDIT", "AMT_GOODS_PRICE"}.issubset(df.columns):
        df["LTV_PROXY"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)

    if "DAYS_BIRTH" in df.columns:
        df["AGE"] = -df["DAYS_BIRTH"] / 365

    if "DAYS_EMPLOYED" in df.columns:
        df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365

    if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(df.columns):
        df["ANNUITY_TO_CREDIT"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"].replace(0, np.nan)

    if {"AMT_INCOME_TOTAL", "CNT_CHILDREN"}.issubset(df.columns):
        df["INCOME_PER_CHILD"] = df["AMT_INCOME_TOTAL"] / (df["CNT_CHILDREN"] + 1)

    if {"EMPLOYMENT_YEARS", "AGE"}.issubset(df.columns):
        df["EMPLOYED_TO_AGE_RATIO"] = df["EMPLOYMENT_YEARS"] / df["AGE"].replace(0, np.nan)

    return df

df = engineer_application_features(df)

## 4. Choose a compact scorecard candidate set

Earlier enhanced model used a larger feature pool and reached **AUC ≈ 0.7461**.  
For a scorecard, it is usually better to move to a **shorter, more explainable set** rather than keep every possible variable.

The list below keeps a mix of:
- application affordability
- leverage
- stability
- bureau behaviour
- repayment conduct
- prior credit behaviour

Revise this list later after reviewing WOE and IV results.

In [ ]:
scorecard_candidates = [
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "INST_TO_INCOME",
    "LTV_PROXY",
    "AGE",
    "EMPLOYMENT_YEARS",
    "ANNUITY_TO_CREDIT",
    "BUREAU_DAYS_CREDIT_MEAN",
    "BUREAU_IS_ACTIVE_SUM",
    "BUREAU_AMT_CREDIT_SUM_MEAN",
    "INSTAL_IS_LATE_MEAN",
    "POS_COUNT",
    "PREV_CNT_PAYMENT_MAX",
    "PREV_IS_REFUSED_MEAN",
]

available_features = [col for col in scorecard_candidates if col in df.columns]
missing_features = [col for col in scorecard_candidates if col not in df.columns]

print("Available scorecard features:")
print(available_features)
print("\nMissing scorecard features:")
print(missing_features)

## 5. Build the modelling dataset

A scorecard should be trained on the same definition of default target used in the PD model.

This notebook uses:
- `TARGET = 1` for default / bad
- `TARGET = 0` for non-default / good

Missing numeric values are left as missing at this stage, because the scorecard will treat missing values as their own bin where needed.

In [ ]:
model_cols = [ID_COL, TARGET] + available_features
model_df = df[model_cols].copy()

# remove rows without target
model_df = model_df[model_df[TARGET].notna()].copy()

print(model_df.shape)
model_df.head()

In [ ]:
train_df, test_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=model_df[TARGET]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nBad rate - train:", train_df[TARGET].mean().round(4))
print("Bad rate - test :", test_df[TARGET].mean().round(4))

## 6. WOE and IV helper functions

A traditional scorecard usually does not feed raw variables directly into logistic regression.

Instead, the common process is:

1. bin variables  
2. calculate **WOE**  
3. review **IV**  
4. fit logistic regression on WOE-transformed variables  

This structure is easier to explain, monitor, and govern.

WOE = Weight of Evidence :Is this group of customers more risky or less risky than average?
WOE Method: split a variable into bins.to find if this group have more bad customers than expected?
Or fewer bad customers than expected?If more bads than goods → risky → negative WOE,If more goods than bads → safe → positive WOE,Near 0 WOE mean neutral group.

WOE Purpose:
- converts categories into numeric values
- makes relationships linear (important for logistic regression) .Becasue Logistic regression works best when relationships are smooth and linear
- ensures monotonic risk trend
- improves model stability
- is easy to explain to business / regulators

converts each bin → one number
enforces risk ordering
removes noise
handles missing values cleanly

IV = Information Value : How useful is this variable for predicting default?Does this variable separate good vs bad customers well? Or are good and bad mixed randomly?
IV Purpose: Feature selection

IV	Meaning:
< 0.02	    useless
0.02 – 0.1	weak
0.1 – 0.3	medium
0.3 – 0.5	strong
> 0.5	    suspicious (possible leakage)

IV measures: how different good % and bad % are across bins
-Big separation → high IV → good predictor
-No separation → low IV → useless variable


In [ ]:

#_safe_qcut() takes a numeric pandas Series, removes missing values, and tries to split it into meaningful bins for modelling: if the data has no variation it returns None, if it has only a few unique values it uses pd.cut() to create simple intervals, otherwise it attempts pd.qcut() from the maximum number of bins down to two, returning the first valid categorical binning with at least two groups; if all attempts fail, it safely returns None instead of raising an error.

def _safe_qcut(series, max_bins=5):
    clean = series.dropna()

    if clean.nunique() <= 1:
        return None

    if clean.nunique() <= max_bins:
        try:
            cats = pd.cut(clean, bins=clean.nunique(), duplicates="drop")
            return cats
        except Exception:
            return None

    for q in range(max_bins, 1, -1):
        try:
            cats = pd.qcut(clean, q=q, duplicates="drop")
            if getattr(cats, "cat", None) is not None and len(cats.cat.categories) >= 2:
                return cats
        except Exception:
            continue
    return None

#If the variable is numeric, try to create bins using _safe_qcut. If binning fails, treat it as categorical by converting values to strings and marking missing as "MISSING". If binning succeeds, assign each value to its bin, convert bins to strings, extract the bin boundaries (edges), and store them in spec for future use. If the variable is not numeric, treat it as categorical by converting all values to strings and filling missing values with "MISSING".

def fit_woe_binning(train_series, target, feature_name, max_bins=5, min_bin_size=0.05, smoothing=0.5):
    temp = pd.DataFrame({"x": train_series, "target": target}).copy()

    if pd.api.types.is_numeric_dtype(temp["x"]):
        binned = _safe_qcut(temp["x"], max_bins=max_bins)
        if binned is None:
            temp["bin"] = temp["x"].astype("string").fillna("MISSING")
            spec = {"type": "categorical"}
        else:
            temp.loc[binned.index, "bin"] = binned.astype(str)
            temp["bin"] = temp["bin"].astype("string").fillna("MISSING")
            intervals = pd.IntervalIndex(binned.cat.categories)
            edges = [intervals[0].left] + [iv.right for iv in intervals]
            spec = {"type": "numeric", "edges": edges}
    else:
        temp["bin"] = temp["x"].astype("string").fillna("MISSING")
        spec = {"type": "categorical"}

    grp = (
        temp.groupby("bin", dropna=False)["target"]
        .agg(total="count", bad="sum")
        .reset_index()
    )
    
    grp["good"] = grp["total"] - grp["bad"]

    # merge tiny bins into MISSING-like fallback is skipped here for clarity;
    # manual review can be added later if needed.

    total_good = grp["good"].sum()
    total_bad = grp["bad"].sum()

    grp["dist_good"] = (grp["good"] + smoothing) / (total_good + smoothing * len(grp))
    grp["dist_bad"] = (grp["bad"] + smoothing) / (total_bad + smoothing * len(grp))
    grp["woe"] = np.log(grp["dist_good"] / grp["dist_bad"])
    grp["iv_component"] = (grp["dist_good"] - grp["dist_bad"]) * grp["woe"]
    grp["iv"] = grp["iv_component"].sum()
    grp["feature"] = feature_name

    mapping = dict(zip(grp["bin"].astype(str), grp["woe"])) # bin label: woe value
    return grp[["feature", "bin", "total", "good", "bad", "dist_good", "dist_bad", "woe", "iv_component", "iv"]], spec, mapping #grp[...] is woe table and spec: binning rules, map format is: bin:, woe: 

#conver the raw value in each feature to bins, like income $3000 bin: (0,5000].
def apply_bins(series, spec):
    if spec["type"] == "numeric":
        edges = spec["edges"]
        binned = pd.cut(series, bins=edges, include_lowest=True)
        return binned.astype("string").fillna("MISSING")
    return series.astype("string").fillna("MISSING")

#converts the given raw dataset into a WOE-transformed dataset using the saved binning rules and mappings.
#format is bin: woe:
def transform_to_woe(df_in, binning_store, default_woe=0.0):
    out = pd.DataFrame(index=df_in.index)

    for feature, meta in binning_store.items():
        bins = apply_bins(df_in[feature], meta["spec"]).astype(str)
        out[feature] = bins.map(meta["mapping"]).fillna(default_woe) #“For each value in bins series, find it in mapping dictionary and replace it”

    return out

## 7. Fit WOE bins and review IV

As a starting point:
- **IV < 0.02** usually means weak predictive value
- **IV 0.02 to 0.10** is weak to medium
- **IV 0.10 to 0.30** is medium to strong
- **IV > 0.30** can be very strong, but should be checked carefully for leakage or instability

This notebook keeps the process transparent so you can manually review the scorecard shortlist.

a complete transformation rulebook for all features:
binning_store = {
    "income": {
        "spec": {
            "type": "numeric",
            "edges": [0, 5000, 10000, 20000]
        },
        "mapping": {
            "(0, 5000]": -0.8,
            "(5000, 10000]": 0.1,
            "(10000, 20000]": 0.9,
            "MISSING": 0.0
        }
    },

    "age": {
        "spec": {
            "type": "numeric",
            "edges": [18, 30, 50, 80]
        },
        "mapping": {
            "(18, 30]": -0.3,
            "(30, 50]": 0.2,
            "(50, 80]": 0.5,
            "MISSING": 0.0
        }
    }
}

later :
apply_bins(df["income"], meta["spec"])#convert to WOE
bins.map(meta["mapping"]) #convert to WOE
apply same transformation to new data
keep model consistent
separate training logic from scoring logic

In [ ]:
binning_store = {}
woe_tables = []

for feature in available_features:
    table, spec, mapping = fit_woe_binning(
        train_df[feature],
        train_df[TARGET],
        feature_name=feature,
        max_bins=5,
        smoothing=0.5,
    )

    woe_tables.append(table)
    #stores the binning rules and WOE mapping for one feature into a dictionary called binning_store,feature is the key of 1 dictionary like above income or agae.
    binning_store[feature] = {"spec": spec, "mapping": mapping}

print(woe_tables[:3])

woe_table = pd.concat(woe_tables, ignore_index=True) #one master WOE table across all features
print(woe_table[:3])

iv_summary = (
    woe_table.groupby("feature", as_index=False)["iv"]
    .max()
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

iv_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(iv_summary["feature"], iv_summary["iv"])
plt.gca().invert_yaxis()
plt.title("Information Value by Scorecard Candidate")
plt.xlabel("Information Value")
plt.tight_layout()
plt.show()

## 8. Finalise scorecard variables

Here the model keeps variables that:
- are present in the dataset
- have usable WOE bins
- have at least modest IV
- remain explainable for portfolio use

You can tighten or relax the IV cut later, but for a GitHub portfolio this provides a sensible first governed shortlist.

In [ ]:
IV_MIN = 0.02

final_scorecard_features = (
    iv_summary.loc[iv_summary["iv"] >= IV_MIN, "feature"]
    .tolist()
)

print("Final scorecard features:")
print(final_scorecard_features)
print("\nNumber of variables:", len(final_scorecard_features))

## 9. Transform train and test sets to WOE

Once bins are defined on the training sample, the same bin rules are applied to the test sample.

This is important because scorecards should be fitted and validated using a stable transformation process.

In [ ]:
#Loop through all items in binning_store, and keep only those where the feature (k) is in final_scorecard_features
final_binning_store = {k: v for k, v in binning_store.items() if k in final_scorecard_features}

X_train_woe = transform_to_woe(train_df[final_scorecard_features], final_binning_store)
X_test_woe = transform_to_woe(test_df[final_scorecard_features], final_binning_store)

y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

X_train_woe.head()

## 10. Fit the scorecard logistic regression

This is the core statistical engine of the scorecard.

The difference from a plain raw-variable logistic model is that the inputs are now **WOE-transformed bins**, which makes the model much more scorecard-like.

In [ ]:
scorecard_model = LogisticRegression(
    max_iter=2000,
    solver="liblinear",
    random_state=RANDOM_STATE
)

scorecard_model.fit(X_train_woe, y_train)

train_pd = scorecard_model.predict_proba(X_train_woe)[:, 1]
test_pd = scorecard_model.predict_proba(X_test_woe)[:, 1]

train_auc = roc_auc_score(y_train, train_pd)
test_auc = roc_auc_score(y_test, test_pd)

print("Train AUC:", round(train_auc, 4))
print("Test AUC :", round(test_auc, 4))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_pd)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"Scorecard ROC (AUC = {test_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - PD Scorecard")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Review coefficient directions

At this stage, you should review whether the model behaves in a business-sensible way.

Typical examples:
- higher repayment burden should usually be riskier
- worse repayment conduct should usually be riskier
- stronger external scores should usually be safer

This is one reason scorecards are attractive in banking: they are easier to inspect than more complex models.

Coefficient:Coefficients (β) measure how each feature affects the log-odds of default - If this feature increases by 1 unit, how does log-odds change?

beta_income = -0.3 - Increasing income reduces log-odds by 0.3
β value	Meaning
β > 0	increases risk
β < 0	decreases risk
β = 0	no effect

Odds Ratio, convert from above beta by np.exp(0.5) ≈ 1.65 - odds ↑ 65%

In [ ]:
coef_table = pd.DataFrame({
    "feature": final_scorecard_features,
    "coefficient": scorecard_model.coef_[0],
})

coef_table["odds_ratio"] = np.exp(coef_table["coefficient"])
coef_table["direction"] = np.where(
    coef_table["coefficient"] > 0,
    "Higher default risk",
    "Lower default risk"
)

coef_table = coef_table.sort_values("coefficient", ascending=False).reset_index(drop=True)
coef_table

## 12. Convert the model into a points-based scorecard
1. sklearn logistic regression result: log(PD / 1-PD)
That is bad / good

log(bad/good or default_odds) = β0 + β1·x1 + β2·x2 + ...- the log(odds)result are hard to explain 

convert to a SCORE -higher score = lower risk and lower score = higher risk
Score is a linear transformation of log(default_odds)
Define :Score = A - B × log(default_odds) - linear transformation preserves ordering
higher PD → higher logit → lower score
lower PD → lower logit → higher score


Rename A and B
A = Offset  
B = Factor
Score = Offset - Factor × log(odds)

If odds double:
log(odds_new) = log(odds_old × 2)
              = log(odds_old) + log(2)

Score = A - B * (log(odds_old)- log(2)) vs Score = A - B * (log(odds_old)) 

Score difference:

ΔScore = B*log(2) = Factor × log(2)   
ΔScore = PDO
PDD = Factor * log(2)
Factor = PDO / ln(2) - when Provide PDO, factor will be solved and factor controls how fast score changes

When provide basescore:
BaseScore = Offset - Factor × log(BaseDefaultOdds)
Offset = BaseScore - Factor * log (BaseDefaultOdds) 

Final score formula:
# Score = scaled - shifted version of log(default odds)

Score = Offset - Factor × log(default odds) when applying above offset, factor, and log (odds) 
Score = BaseScore-Factor * log(BaseDefaultOdds)-Factor * log(default odds)
Score = BaseScore-Factor * (log(basedefaultodds+log(default odds)))


- **base odds** = 0.08  # average P(default) = 8% from given historical data
- **base score** = 600, The score assigned to the above base default odds above, when base default odd = 8%, we decide the base score is 600
- **PDO** = 20 points to double the odds ,Points to Double the Odds,How many points are needed to double the odds,+20 points → odds × 2 (better borrower) AND -20 points → odds ÷ 2 (worse borrower)

Score = 600 → default odds = 8%
Score = 620 → default odds = 16%
Score = 580 → default odds = 4%

In [ ]:
avg_pd = train_df[TARGET].mean().round(2) # average default rate P (Default)
good_odds = (1 - avg_pd).round(2)
print(avg_pd, good_odds)

In [ ]:
BASE_SCORE = 600
BASE_ODDS = 0.08
PDO = 20

FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

n_features = len(final_scorecard_features)

#logit=β0​+β1​x1​+…from scikit-learn logistic regression, and intercept_[0] = 𝛽0 (The model’s prediction when all features = 0,logit = intercept + feature effects), Intercept contribution = −FACTOR × β0​, so below are baseline score of the model

base_points_per_feature = (OFFSET - FACTOR * scorecard_model.intercept_[0]) / n_features

print("Factor:", round(FACTOR, 4))
print("Offset:", round(OFFSET, 4))
print("Base points allocated per feature:", round(base_points_per_feature, 4))

Final Scorecard :Total Score = sum(points from each feature bin)
logit = β0 ​+ β1​* Xi​+…

After WOE transformation:
xi​=WOEi​

so 
logit = β0 ​+ ∑ βi​⋅WOEi​

Convert to score:

Score = Offset− Factor × logit
Substitute:

Score = Offset − Factor⋅β0​−Factor⋅∑βi​⋅WOEi​

Now each feature contributes: −Factor⋅βi​⋅WOEi​

points = base_points_per_feature - FACTOR * beta * WOE

In [ ]:
scorecard_points_rows = []

coef_lookup = dict(zip(final_scorecard_features, scorecard_model.coef_[0]))

for feature in final_scorecard_features:
    feature_table = woe_table[woe_table["feature"] == feature].copy()
    beta = coef_lookup[feature]
    feature_table["beta"] = beta
    feature_table["points"] = base_points_per_feature - FACTOR * beta * feature_table["woe"]
    scorecard_points_rows.append(feature_table)

scorecard_points = pd.concat(scorecard_points_rows, ignore_index=True)
scorecard_points = scorecard_points.sort_values(["feature", "bin"]).reset_index(drop=True)

scorecard_points[["feature", "bin", "woe", "points", "iv"]].head(20)

## 13. Score individual borrowers

Now we can score each borrower on:
- total score
- predicted PD
- score band

This is the step where modelling starts becoming useful for **strategy, monitoring, and credit policy**.

In [ ]:
points_lookup = {
    feature: dict(zip(
        scorecard_points.loc[scorecard_points["feature"] == feature, "bin"].astype(str),
        scorecard_points.loc[scorecard_points["feature"] == feature, "points"],
    ))
    for feature in final_scorecard_features
}

def score_observations(df_in, binning_store, points_lookup, model, features):
    scored = df_in[[ID_COL, TARGET] + features].copy()

    total_points = np.zeros(len(scored))
    transformed = transform_to_woe(scored[features], binning_store)
    pd_pred = model.predict_proba(transformed)[:, 1]

    for feature in features:
        bins = apply_bins(scored[feature], binning_store[feature]["spec"]).astype(str)
        scored[f"{feature}__bin"] = bins
        scored[f"{feature}__points"] = bins.map(points_lookup[feature]).fillna(0.0)
        total_points += scored[f"{feature}__points"].values

    scored["score"] = total_points
    scored["pd_pred"] = pd_pred
    return scored

train_scored = score_observations(train_df, final_binning_store, points_lookup, scorecard_model, final_scorecard_features)

test_scored = score_observations(test_df, final_binning_store, points_lookup, scorecard_model, final_scorecard_features)

print("Train Scored:")
print(train_scored[["score", "pd_pred", TARGET]].head())
print("\nTest Scored:")
print(test_scored[["score", "pd_pred", TARGET]].head())

In [ ]:
def assign_score_band(score_series):
    bins = [-np.inf, 540, 580, 620, 660, np.inf]
    labels = ["E", "D", "C", "B", "A"]
    return pd.cut(score_series, bins=bins, labels=labels)

train_scored["score_band"] = assign_score_band(train_scored["score"])
test_scored["score_band"] = assign_score_band(test_scored["score"])

test_scored[["score", "score_band", "pd_pred", TARGET]].head()

## 14. Save scorecard build artefacts

These outputs are useful both for GitHub presentation and for the follow-up validation notebook.

In [ ]:
iv_summary.to_csv(OUTPUT_DIR / "01_iv_summary.csv", index=False)
woe_table.to_csv(OUTPUT_DIR / "02_woe_table.csv", index=False)
coef_table.to_csv(OUTPUT_DIR / "03_scorecard_coefficients.csv", index=False)
scorecard_points.to_csv(OUTPUT_DIR / "04_scorecard_points.csv", index=False)
train_scored.to_csv(OUTPUT_DIR / "05_train_scored.csv", index=False)
test_scored.to_csv(OUTPUT_DIR / "06_test_scored.csv", index=False)

scorecard_metadata = pd.DataFrame({
    "metric": ["train_auc", "test_auc", "base_score", "base_odds", "pdo", "n_features"],
    "value": [train_auc, test_auc, BASE_SCORE, BASE_ODDS, PDO, len(final_scorecard_features)]
})
scorecard_metadata.to_csv(OUTPUT_DIR / "07_scorecard_metadata.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())
sorted([p.name for p in OUTPUT_DIR.iterdir()])

## 15. Notebook wrap-up

At this point you have a proper scorecard workflow:
- compact candidate set
- WOE bins
- IV review
- logistic regression on WOE features
- score points
- borrower scores and score bands

The next notebook will focus on:
- validation
- decile analysis
- KS / Gini
- calibration
- approval strategy and business interpretation